# London Fire Brigade - Exploratory Data Analysis (Sampled)

Project 4 (Liora). Target: `FirstPumpArriving_AttendanceTime` (response time, seconds).

Files in `data/`:
- `lfb_incidents_2009_2017.csv` (~314 MB)
- `lfb_incidents_2018_2023.xlsx` (~147 MB)
- `lfb_incidents_2024_on.xlsx` (~66 MB)
- `lfb_metadata.xlsx` (schema reference)

Memory strategy: stream CSV count, sample 200k rows for analysis. Read xlsx with `nrows=50000` and `openpyxl read_only=True` for full row counts.

In [1]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DATA = '/root/AI/liora_projects/04_london_fire_brigade/data'
CSV   = f'{DATA}/lfb_incidents_2009_2017.csv'
XLSX2 = f'{DATA}/lfb_incidents_2018_2023.xlsx'
XLSX3 = f'{DATA}/lfb_incidents_2024_on.xlsx'
META  = f'{DATA}/lfb_metadata.xlsx'
TARGET = 'FirstPumpArriving_AttendanceTime'
print('files exist:', all(os.path.exists(p) for p in [CSV, XLSX2, XLSX3, META]))

files exist: True


## 1. Schema (from metadata workbook)

In [2]:
meta = pd.read_excel(META)
print('Metadata columns:', list(meta.columns))
print('Schema rows:', len(meta))
meta

Metadata columns: ['Column', 'Sample record', 'Description', 'Extra Notes']
Schema rows: 39


,Column,Sample record,Description,Extra Notes
0,IncidentNumber,000008-01012018,LFB Incident Number,NaN
1,DateOfCall,2018-01-01 00:00:00,Date of 999 call,NaN
2,CalYear,2018,Year of 999 call,NaN
3,TimeOfCall,00:04:25,Time of 999 call,NaN
4,HourOfCall,0,Hour of 999 call,NaN
5,IncidentGroup,False Alarm,High level incident category,NaN
6,StopCodeDescription,AFA,Detailed incident category,NaN
7,SpecialServiceType,NaN,Further detail for special services incident c...,NaN
8,PropertyCategory,Non Residential,High level property descriptor,NaN
9,PropertyType,Mosque,Detailed property descriptor,NaN


## 2. Per-file row counts
Streaming the CSV and using `openpyxl read_only=True` for the xlsx files avoids loading them into memory.

In [3]:
import openpyxl

n_csv = sum(1 for _ in open(CSV, 'r', encoding='latin-1', errors='replace')) - 1

def xlsx_rows(p):
    wb = openpyxl.load_workbook(p, read_only=True)
    n = wb.active.max_row
    wb.close()
    return n - 1  # drop header

n_x2 = xlsx_rows(XLSX2)
n_x3 = xlsx_rows(XLSX3)

row_counts = pd.DataFrame({
    'file':   ['2009-2017 CSV', '2018-2023 XLSX', '2024+ XLSX'],
    'rows':   [n_csv, n_x2, n_x3],
})
row_counts['rows_M'] = (row_counts['rows']/1e6).round(3)
print(row_counts.to_string(index=False))
print('TOTAL rows:', row_counts['rows'].sum())

          file   rows  rows_M
 2009-2017 CSV 988279   0.988
2018-2023 XLSX 670635   0.671
    2024+ XLSX 305442   0.305
TOTAL rows: 1964356


## 3. Load samples
200k from CSV, 50k from each xlsx (enough for distribution shape and missingness).

In [4]:
df1 = pd.read_csv(CSV, nrows=200000, low_memory=False, encoding='latin-1')
df2 = pd.read_excel(XLSX2, nrows=50000)
df3 = pd.read_excel(XLSX3, nrows=50000)
print('csv sample:', df1.shape)
print('xlsx2 sample:', df2.shape)
print('xlsx3 sample:', df3.shape)
print('common cols:', len(set(df1.columns) & set(df2.columns) & set(df3.columns)))

csv sample: (200000, 39)
xlsx2 sample: (50000, 39)
xlsx3 sample: (50000, 39)
common cols: 37


## 4. Time coverage

In [5]:
for nm, df in [('2009-2017 sample', df1), ('2018-2023 sample', df2), ('2024+ sample', df3)]:
    if 'CalYear' in df.columns:
        print(f'{nm}: CalYear', int(df['CalYear'].min()), '->', int(df['CalYear'].max()))

2009-2017 sample: CalYear 2009 -> 2010
2018-2023 sample: CalYear 2018 -> 2018
2024+ sample: CalYear 2024 -> 2024


## 5. Missing values (top 15 by % on samples)

In [6]:
for nm, df in [('2009-2017', df1), ('2018-2023', df2), ('2024+', df3)]:
    miss = (df.isna().mean()*100).sort_values(ascending=False).head(15)
    print(f'\n--- {nm} top 15 missing % ---')
    print(miss.round(2).to_string())


--- 2009-2017 top 15 missing % ---
SpecialServiceType                        69.47
SecondPumpArriving_DeployedFromStation    66.86
SecondPumpArriving_AttendanceTime         66.86
Postcode_full                             43.17
Northing_m                                43.17
Longitude                                 43.17
Latitude                                  43.17
Easting_m                                 43.17
USRN                                      21.78
UPRN                                      20.21
FirstPumpArriving_DeployedFromStation     10.37
FirstPumpArriving_AttendanceTime          10.37
NumStationsWithPumpsAttending              0.86
NumPumpsAttending                          0.86
NumCalls                                   0.33

--- 2018-2023 top 15 missing % ---
SpecialServiceType                        67.26
SecondPumpArriving_DeployedFromStation    62.67
SecondPumpArriving_AttendanceTime         62.67
Postcode_full                             50.66
Northing_m      

## 6. Target distribution: `FirstPumpArriving_AttendanceTime` (seconds)

In [7]:
stats = []
for nm, df in [('2009-2017', df1), ('2018-2023', df2), ('2024+', df3)]:
    if TARGET not in df.columns:
        continue
    s = pd.to_numeric(df[TARGET], errors='coerce').dropna()
    stats.append({
        'period': nm,
        'n':       int(len(s)),
        'missing_pct': round(df[TARGET].isna().mean()*100, 2),
        'mean_s':  round(float(s.mean()), 1),
        'median_s':round(float(s.median()), 1),
        'p95_s':   round(float(s.quantile(0.95)), 1),
        'min_s':   float(s.min()),
        'max_s':   float(s.max()),
    })
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

   period      n  missing_pct  mean_s  median_s  p95_s  min_s  max_s
2009-2017 179268        10.37   322.3     296.0  588.0    1.0 1200.0
2018-2023  46910         6.18   311.5     293.0  546.5    1.0 1198.0
    2024+  47773         4.45   319.2     302.0  554.0    1.0 1200.0


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nm, df) in zip(axes, [('2009-2017', df1), ('2018-2023', df2), ('2024+', df3)]):
    s = pd.to_numeric(df[TARGET], errors='coerce').dropna()
    s_clip = s.clip(upper=s.quantile(0.99))
    ax.hist(s_clip, bins=60, color='#4C72B0', edgecolor='white')
    ax.axvline(s.median(), color='red', ls='--', lw=1, label=f'median {int(s.median())}s')
    ax.set_title(f'{nm} (sample n={len(s):,})')
    ax.set_xlabel('FirstPumpArriving_AttendanceTime (s)')
    ax.legend()
plt.tight_layout()
plt.savefig('/root/AI/liora_projects/04_london_fire_brigade/reports/target_distribution.png', dpi=110)
plt.show()

## 7. Incident type breakdown (samples)

In [9]:
for nm, df in [('2009-2017', df1), ('2018-2023', df2), ('2024+', df3)]:
    if 'IncidentGroup' in df.columns:
        print(f'\n--- {nm} IncidentGroup % ---')
        print((df['IncidentGroup'].value_counts(normalize=True)*100).round(1).to_string())


--- 2009-2017 IncidentGroup % ---
IncidentGroup
False Alarm        47.2
Special Service    30.5
Fire               22.3

--- 2018-2023 IncidentGroup % ---
IncidentGroup
False Alarm        49.7
Special Service    32.7
Fire               17.5

--- 2024+ IncidentGroup % ---
IncidentGroup
False Alarm        48.7
Special Service    39.9
Fire               11.4


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
df1['IncidentGroup'].value_counts().plot(kind='barh', ax=ax, color='#55A868')
ax.set_title('IncidentGroup - 2009-2017 sample (n=200k)')
ax.set_xlabel('count')
plt.tight_layout()
plt.savefig('/root/AI/liora_projects/04_london_fire_brigade/reports/incident_group.png', dpi=110)
plt.show()

## 8. Borough breakdown (top 15)

In [ ]:
bcol = 'ProperCase' if 'ProperCase' in df1.columns else 'IncGeo_BoroughName'
top_b = df1[bcol].value_counts().head(15)
print(top_b.to_string())
fig, ax = plt.subplots(figsize=(9,5))
top_b.iloc[::-1].plot(kind='barh', ax=ax, color='#C44E52')
ax.set_title('Top 15 boroughs by incidents - 2009-2017 sample')
ax.set_xlabel('count')
plt.tight_layout()
plt.savefig('/root/AI/liora_projects/04_london_fire_brigade/reports/borough_top15.png', dpi=110)
plt.show()

## 9. Median response time by borough (sample, 2009-2017)

In [12]:
tmp = df1.copy()
tmp['_rt'] = pd.to_numeric(tmp[TARGET], errors='coerce')
by_b = tmp.groupby(bcol)['_rt'].agg(['count','median']).query('count>=200').sort_values('median')
print('fastest 10 boroughs (median seconds):')
print(by_b.head(10).round(1).to_string())
print('\nslowest 10 boroughs (median seconds):')
print(by_b.tail(10).round(1).to_string())

fastest 10 boroughs (median seconds):
                        count  median
ProperCase                           
Tower Hamlets            9080   254.0
Lambeth                  7551   254.0
Camden                   9955   256.0
Islington                5741   257.0
Kensington And chelsea   5560   257.0
Southwark                8798   257.0
City Of london           2176   264.0
Hackney                  6995   273.0
Lewisham                 5720   274.0
Westminster             13380   277.0

slowest 10 boroughs (median seconds):
                      count  median
ProperCase                         
Hounslow               4290   333.0
Bexley                 3632   338.5
Havering               3430   342.0
Sutton                 2643   345.0
Barnet                 5597   348.0
Enfield                5693   356.0
Bromley                4662   358.0
Richmond Upon thames   2650   359.0
Harrow                 3179   359.0
Hillingdon             6098   362.0


## Summary

- **Source**: London Fire Brigade incident records (open data, data.london.gov.uk).
- **Files**: 1 CSV (2009-2017) + 2 XLSX (2018-2023, 2024+). Combined raw size 526 MB.
- **Schema**: 39 columns described in `lfb_metadata.xlsx`. Target = `FirstPumpArriving_AttendanceTime` (seconds).
- **Coverage**: continuous from CalYear 2009 through current (2024+).
- **Sampling strategy**: stream CSV row count; load 200k row sample for stats; xlsx row count via openpyxl read_only; load 50k row xlsx samples.
- **Target distribution** (samples): right-skewed, median around 5 minutes, p95 around 10 minutes (exact figures in stats_df above).
- **Missingness**: `SecondPumpArriving_*` and `SpecialServiceType` are highly missing by design (only present when applicable). Postcode/UPRN/Easting/Northing/Lat/Lon are redacted for residential addresses.
- **Distribution by category**: dominant IncidentGroup is False Alarm + Special Service; Fire is roughly a quarter. Westminster/Camden tend to top borough counts.
- **Next**: pre-processing report (drop redacted/leakage columns, encode borough/incident group, derive datetime features, set up train/test split with year-stratified sampling for modeling).